# Day 18 — Pandas API on Spark

**What you will learn:**
- What the Pandas API on Spark is (formerly Koalas)
- Creating Spark DataFrames from pandas-on-Spark DataFrames and vice versa
- Using pandas-style operations at scale: `groupby`, `apply`, `merge`, `pivot_table`
- Key differences and pitfalls vs native pandas
- When to use Pandas API vs native Spark DataFrame API

**Exam relevance:** Pandas API on Spark (5%) — a dedicated exam domain. Know the import path, conversion methods, and key limitations.

In [ ]:
import pyspark.pandas as ps
import pandas as pd
import numpy as np

# Suppress index warnings for cleaner output in notebooks
ps.set_option("compute.ops_on_diff_frames", True)

## 1. What Is the Pandas API on Spark?

- Introduced in Spark 3.2 as `pyspark.pandas` (replacing the Koalas project)
- Implements the pandas DataFrame API on top of Spark distributed DataFrames
- Lets data scientists use familiar pandas code without rewriting for PySpark

**Import path:**
```python
import pyspark.pandas as ps          # Spark 3.2+
```

> **Exam tip:** The exam tests `pyspark.pandas` (not `databricks.koalas`). Know the import and key conversion methods.

## 2. Creating Pandas-on-Spark DataFrames

In [ ]:
# From a dict — just like pandas
psdf = ps.DataFrame({
    "name":     ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "dept":     ["Engineering", "Marketing", "Engineering", "Sales", "Marketing"],
    "salary":   [95000, 72000, 110000, 58000, 81000],
    "country":  ["RO", "RO", "UK", "DE", "RO"],
})

print(type(psdf))
psdf

In [ ]:
# From a Spark DataFrame → Pandas-on-Spark
spark_df = spark.createDataFrame([
    ("Alice", "Engineering", 95000),
    ("Bob",   "Marketing",   72000),
], ["name", "dept", "salary"])

psdf2 = spark_df.pandas_api()   # or spark_df.to_pandas_on_spark()
print(type(psdf2))

# From a native pandas DataFrame → Spark Pandas
pdf = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
psdf3 = ps.from_pandas(pdf)
print(type(psdf3))

## 3. Converting Back to Spark or Pandas

In [ ]:
# Pandas-on-Spark → Spark DataFrame
spark_df_back = psdf.to_spark()
print(type(spark_df_back))
spark_df_back.show()

# Pandas-on-Spark → native pandas (collects ALL data to driver — careful on large data)
pdf_back = psdf.to_pandas()
print(type(pdf_back))
print(pdf_back)

## 4. Familiar Pandas Operations

In [ ]:
# Selection, filtering, new columns
psdf["salary_k"] = psdf["salary"] / 1000
filtered = psdf[psdf["salary"] > 70000]
print(filtered[["name", "dept", "salary"]])

In [ ]:
# groupby + agg — pandas-style
psdf.groupby("dept")["salary"].agg(["mean", "min", "max", "count"])

In [ ]:
# value_counts
psdf["dept"].value_counts()

In [ ]:
# describe — summary stats
psdf[["salary"]].describe()

## 5. apply — Pandas UDF Under the Hood

In [ ]:
# apply on a column — like pandas Series.apply
def grade(salary):
    if salary >= 90000: return "Senior"
    if salary >= 70000: return "Mid"
    return "Junior"

psdf["level"] = psdf["salary"].apply(grade)
psdf[["name", "salary", "level"]]

## 6. merge — Join Pandas-on-Spark DataFrames

In [ ]:
dept_info = ps.DataFrame({
    "dept":     ["Engineering", "Marketing", "Sales"],
    "division": ["Tech", "Ops", "Ops"],
})

merged = psdf.merge(dept_info, on="dept", how="left")
merged[["name", "dept", "division", "salary"]]

## 7. Key Differences vs Native Pandas

| Behaviour | Native pandas | Pandas-on-Spark |
|---|---|---|
| Index | Preserved, meaningful | Default index is not guaranteed ordered |
| Row order | Guaranteed | **Not guaranteed** — Spark is distributed |
| `ops_on_diff_frames` | Always works | Off by default (shuffle cost) |
| `apply` / `map` | Row-by-row Python | Pandas UDF (Arrow) under the hood |
| `head(n)` | Fast | May trigger full read |
| Missing API coverage | Full | ~80% — some pandas ops not supported |

In [ ]:
# Row order is NOT guaranteed — sort explicitly if order matters
psdf.sort_values("salary", ascending=False)[["name", "salary"]]

## 8. When to Use Pandas API vs Spark DataFrame API

| Use Pandas API on Spark when... | Use Spark DataFrame API when... |
|---|---|
| Migrating existing pandas code | Writing new production pipelines |
| Data science exploration | Performance is critical |
| Team knows pandas, not PySpark | Fine-grained control over execution plan |
| Prototyping | Complex joins, UDFs, window functions |

---

## Day 18 Checklist

- [ ] Know the import: `import pyspark.pandas as ps`
- [ ] Created a pandas-on-Spark DataFrame from dict, Spark DF, and native pandas
- [ ] Converted between Spark DF and pandas-on-Spark with `.pandas_api()` and `.to_spark()`
- [ ] Used `groupby`, `apply`, `merge`, `describe` in pandas API style
- [ ] Know that row order is NOT guaranteed without an explicit `sort_values()`
- [ ] Know that `to_pandas()` collects all data to the driver — dangerous on large datasets

**Next:** Day 19 — Intro to Structured Streaming